In [ ]:
# The pre-requisite for this python file is ncat to be installed 
#docker exec -it <jupyter-lab-process-id>
#sudo apt-get install ncat

In [1]:
from pyspark.sql import SparkSession

spark=SparkSession\
.builder \
.appName("read from socket") \
.master("local[*]") \
.getOrCreate()

spark

In [2]:
#Read Input
df_raw=spark.readStream.format("socket").option("host","localhost").option("port","9999").load()
# df_raw.printSchema()

In [3]:
#
from pyspark.sql.functions import split

df_splt=df_raw.withColumn("words",split("value"," "))
# df_splt.show()

In [4]:
#explode
from pyspark.sql.functions import explode

df_explode=df_splt.withColumn("word",explode("words")).drop("value","words")
# df_explode.show()

In [5]:
from pyspark.sql.functions import lit,count
df_agg=df_explode.groupBy("word").agg(count(lit (1)).alias("CNT"))
# df_agg.show()

In [6]:
#WriteStream
df_agg.writeStream.format("console").outputMode("complete").start().awaitTermination

<bound method StreamingQuery.awaitTermination of <pyspark.sql.streaming.StreamingQuery object at 0x7f897c3218d0>>